In [4]:
# Import required libraries
import pandas as pd
import numpy as np

# Set random seed for reproducibility
np.random.seed(42)

# Define number of samples
n_samples = 20000  # Already increased to ensure more default samples

# Generate synthetic credit risk dataset
data = {
    'customer_id': range(1, n_samples + 1),
    'loan_amount': np.random.normal(50000, 15000, n_samples).clip(min=1000, max=500000),  # Loan amount in EUR
    'loan_term': np.random.choice([12, 24, 36, 48, 60], n_samples),  # Loan term in months
    'credit_score': np.random.normal(700, 50, n_samples).clip(min=300, max=850),  # Credit score (300-850)
    'income': np.random.normal(60000, 20000, n_samples).clip(min=5000, max=500000),  # Annual income in EUR
    'employment_status': np.random.choice(['Full-time', 'Part-time', 'Self-employed', 'Unemployed'], n_samples),
    'debt_to_income': np.random.uniform(0, 1, n_samples),  # Debt-to-income ratio (0-1)
}

# Create DataFrame
df = pd.DataFrame(data)

# Generate default probability based on features
# Higher credit score -> lower default probability
# Higher debt-to-income -> higher default probability
df['default_probability'] = (
    0.2 * (1 - (df['credit_score'] - 300) / (850 - 300)) +  # Credit score effect (scaled 0-1)
    0.5 * df['debt_to_income'] +  # Debt-to-income effect
    0.1 * (df['loan_amount'] / 50000) +  # Loan amount effect
    0.1 * (df['income'] < 30000)  # Low income effect
)

# Clip default probability to [0, 1]
df['default_probability'] = df['default_probability'].clip(0, 1)

# Check default probability distribution
print("\n=== Default Probability Distribution ===")
print(df['default_probability'].describe())

# Generate default status based on default probability
# Use a threshold to control default rate (target: 5%-10%)
default_threshold = 0.65  # Lowered from 0.9 to achieve 5%-10% default rate
df['default_status'] = (df['default_probability'] > default_threshold).astype(int)

# Check default rate
default_rate = df['default_status'].mean()
print(f"\nDefault Rate: {default_rate:.3f} ({default_rate * 100:.1f}%)")
print("Number of Defaults:", df['default_status'].sum())

# Save the dataset
df.to_csv('credit_risk_dataset.csv', index=False)
print("Dataset saved as 'credit_risk_dataset.csv'")


=== Default Probability Distribution ===
count    20000.000000
mean         0.411767
std          0.150718
min          0.033031
25%          0.286671
50%          0.411864
75%          0.537304
max          0.817816
Name: default_probability, dtype: float64

Default Rate: 0.044 (4.4%)
Number of Defaults: 883
Dataset saved as 'credit_risk_dataset.csv'
